# mva_08-クラスタ分析(1)

---

## **要点まとめ**

**Part Ⅲ：凝集度の最適化**

これまでの回帰では、正解データ（教師ラベル $y$）が存在し、その誤差を最小化することを目指しました（教師あり学習）。
今回から始まる **クラスタ分析（Cluster Analysis）** は、正解ラベルを持たないデータに対し、データ間の「似ている・似ていない」という関係性だけを頼りにグループ（クラスタ）を作成する 教師なし学習 の手法です。

### **1. クラスタ分析の全体像：行方向の要約**

データを行列（表）の形式で整理すると、これまでの分析との違いが明確になります。クラスタ分析は **「行（サンプル）」を似たもの同士でまとめ、行方向を粗視化（要約）する** アプローチです。

クラスタ分析で扱うデータには、回帰分析のような目的変数の列（$y$）は存在しません。予測対象である正解となる列がないため、データの特徴からグループ構造を発見し、「クラスタ（$C$）」という新しい列（ラベル）を作る ことが目的となります。

|  | 変数1 | ... | 変数d |  | **クラスタC** |
 | ----- | ----- | ----- | ----- | ----- | ----- |
| **サンプル1** | $x_{11}$ | ... | $x_{1d}$ | $\rightarrow$ | $C_1$ |
| **サンプル2** | $x_{21}$ | ... | $x_{2d}$ | $\rightarrow$ | $C_1$ |
| **サンプル3** | $x_{31}$ | ... | $x_{3d}$ | $\rightarrow$ | $C_2$ |
| ... | ... | ... | ... | ... | ... |
| **サンプルn** | $x_{n1}$ | ... | $x_{nd}$ | $\rightarrow$ | $C_K$ |

<br>
例えば、数万人の顧客データ（$n$行）を、そのまま一つ一つ見ることは不可能です。しかし、それを「優良顧客」「離反予備軍」などの数個のグループ（$K$個の行）にまとめることができれば、人間の脳でも理解し、戦略を立てることが可能になります。

<br>

### **2. K-means法（K-平均法）**

**非階層的クラスタリング** の代表的手法である K-means法 について、数理最適化の視点からその仕組みを理解しましょう。

#### **(1) 数理最適化の視点：凝集度の最大化**

良いクラスタリングとは何でしょうか？ 一般にクラスタリングはデータ間の類似度（距離）に基づきますが、K-means法においては、データ同士の距離ではなく、**「各データが所属するクラスタの重心（代表点）といかに近いか」** という視点で類似度を測るのがポイントです。直感的には、重心の周りにデータが「ギュッと集まっている（凝集度が高い）」状態が良い状態と言えます。

これを数学的に表現すると、**「クラスタ内誤差平方和 (SSE: Sum of Squared Errors)」** の最小化問題となります。この問題は「どのデータがどのクラスタに属するか」という **割り当て** $C$ に関する問題であり、以下の目的関数 $J(C)$ を最小化することを目指します。（ただし、$\mathbf{\bar{x}}_k$ はクラスタ $C_k$ の重心（平均ベクトル）とします）

$$J(C) = \sum_{k=1}^{K} \sum_{i \in C_k} ||\mathbf{x}_i - \mathbf{\bar{x}}_k||^2$$

- $K$: クラスタの数
- $C_k$: $k$ 番目のクラスタに属するデータの集合（割り当て）
- $\mathbf{x}_i$: データ点
- $\mathbf{\bar{x}}_k$: クラスタ $k$ の重心（平均ベクトル）

しかし、考えうる $C$ の組み合わせパターンは天文学的な数になり、しらみつぶしに計算して最適解を探すことは事実上不可能です。

<br>

#### **(2) アルゴリズムの視点：交互更新による解決**

そこで、K-means法では、計算を可能にするために重心 $\mathbf{\bar{x}}$ を独立した変数（媒介変数）として扱い、問題を変数 $C$ と $\mathbf{\bar{x}}$ の同時最適化問題へと拡張します。

$$J(C, \mathbf{\bar{x}}) = \sum_{k=1}^{K} \sum_{i \in C_k} ||\mathbf{x}_i - \mathbf{\bar{x}}_k||^2$$

この拡張により、以下の2つのステップを交互に繰り返すことで、効率的に解を探索（準最適化）できるようになります。これが K-means法のアルゴリズムです。

- **初期化**: データ空間上にランダムに $K$ 個の「重心 $\mathbf{\bar{x}}$」を配置する。
- **割り当て (E-Step)**: 重心 $\mathbf{\bar{x}}$ を固定して、最適な所属 $C$ を決める（全てのデータについて最も近い重心を選ぶ）。
- **更新 (M-Step)**: 所属 $C$ を固定して、最適な重心 $\mathbf{\bar{x}}$ を決める（各クラスタの平均位置へ重心を移動させる）。
- **収束**: 重心が動かなくなるまで 2. と 3. を繰り返す。

このように、重心を介在させることで、困難な組み合わせ最適化問題を、シンプルな反復計算に落とし込んでいます。

<br>

### **3. Kの決定：エルボー法**

K-means法の最大の弱点は、「いくつのグループに分けるか（$K$）」を人間が決める必要がある点です。
$K$ を増やせば増やすほど、データと重心の距離（SSE）は必ず減少します（極端な話、データ数と同じだけ重心を用意すれば誤差は0になります）。しかし、これではデータを要約したことになりません。

そこで、**「$K$ を増やしたときの誤差の減り具合」** をグラフにし、ガクッと減らなくなる折れ曲がり地点（エルボー：肘）を最適な $K$ とする **エルボー法** がよく用いられます。




## **Python 演習課題**

### **mva_08-A**

✅ **目的**: K-means法における「割り当て」と「更新」のプロセスを手計算で追体験し、アルゴリズムがどのように重心を移動させるかを数理的に理解します。また、Pythonを用いてその計算結果を確認し、学習プロセス全体をアニメーションで確認しましょう。

#### 🖊 **【数理理解】手計算による更新プロセスの体験**

以下の単純なデータ（4点）と、初期重心（2点）を用いて、K-means法の最初の1ステップ（1往復）がどのように進むか計算してみましょう。
今回は、重心の移動を観察しやすくするため、初期重心をあえてデータから少し離れた位置（左上と右下） に設定しています。

【データ点】<br>
$\mathbf{x_1} = (1, 2),\quad \mathbf{x_2} = (2, 1),\quad \mathbf{x_3} = (4, 4), \quad \mathbf{x_4} = (5, 5)$

【初期重心 (Step 0)】<br>
クラスタAの重心 $\mathbf{\bar{x}}_A = (1, 4)$, クラスタBの重心 $\mathbf{\bar{x}}_B = (5, 1)$

**Step 1. 割り当て (E-Step)**

各データ点について、2つの重心 $\mathbf{\bar{x}}_A, \mathbf{\bar{x}}_B$ までの距離の2乗（$||\mathbf{x} - \mathbf{\bar{x}}||^2$）を計算し、近い方のクラスタに割り当てます。

| データ点 | $\mathbf{\bar{x}}_A(1, 4)$ への距離の2乗 | $\mathbf{\bar{x}}_B(5, 1)$ への距離の2乗 | 所属クラスタ |
| :---: | :---: | :---: | :---: |
| $x_1 (1, 2)$ | $(1-1)^2+(2-4)^2 = 4$ | $(1-5)^2+(2-1)^2 = 17$ | **A** |
| $x_2 (2, 1)$ | $10$ | $9$ | **B** |
| $x_3 (4, 4)$ | $9$ | $10$ | **A** |
| $x_4 (5, 5)$ | $17$ | $16$ | **B** |

<br>

**Step 2. 更新 (M-Step)**

**Step 1**の結果に基づき、各クラスタの新しい重心（平均ベクトル）を計算します。
今回の初期重心の設定では、Step 1の段階ではまだクラスタが混在しており（$x_1, x_3$ がA、$x_2, x_4$ がB）、綺麗なグループ分けにはなっていません。重心を更新することでどうなるか見てみましょう。

- クラスタA (新しい重心 $\mathbf{\bar{x}}'_A$):
$$\mathbf{\bar{x}}'_A=\frac{\mathbf{x_1}+\mathbf{x_3}}{2}=(2.5, 3.0)$$
- クラスタB (新しい重心 $\mathbf{\bar{x}}'_B$):
$$\mathbf{\bar{x}}'_B=\frac{\mathbf{x_2}+\mathbf{x_4}}{2}=(3.5, 3.0)$$


この更新された重心を使って **「次の割り当て（2回目のStep 1）」** を行うと、$x_2$ はAへ、$x_3$ はBへと移動し、最終的に $\{x_1, x_2\}$ と $\{x_3, x_4\}$ という本来のグループに正しく分かれます。

<br>

#### **具体例で確認してみよう！**

以下のコードを実行して、この学習が進んでいく様子をアニメーションで確認しましょう。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ==========================================
# Part 1: 手計算問題のデータ設定
# ==========================================
print("--- 手計算問題のデータ確認 ---")

# データ定義 (x1 ~ x4)
# 更新時に小数が切り捨てられないよう、明示的にfloat型に変換します
X_manual = np.array([
    [1, 2], # x1
    [2, 1], # x2
    [4, 4], # x3
    [5, 5]  # x4
]).astype(float)

# 初期重心 (Step 0)
# 重心が収束する様子（移動する様子）を観察しやすくするため、あえて少し離れた位置に設定します
# A: (1, 4), B: (5, 1)
centroids_manual = np.array([
    [1, 4],
    [5, 1]
]).astype(float)

print(f"データ点:\n{X_manual}")
print(f"初期重心:\n{centroids_manual}")

# ==========================================
# Part 2: 学習プロセスのアニメーション可視化
# ==========================================
# 手計算と同じデータと初期重心を使って、アルゴリズムの挙動を再現します。
# 「割り当て」と「更新」のステップを詳細に記録してアニメーション化します。

class KMeansDetailedHistory:
    def __init__(self, X, K, initial_centroids):
        self.X = X
        self.K = K
        self.centroids = initial_centroids.copy()
        # 履歴: (centroidsの位置, 所属ラベル, ステップの説明)
        self.history = []

    def fit(self, max_iter=10):
        # 0. 初期状態の保存
        # まだ割り当てが決まっていないので便宜上距離計算して色を付ける
        current_labels = self._assign_labels()
        self.history.append((self.centroids.copy(), current_labels, "Step 0: Initialization"))

        for i in range(max_iter):
            # --- Step 1: 割り当て (Assignment) ---
            # 重心は動かさず、色（所属）だけ変える
            old_labels = current_labels
            current_labels = self._assign_labels()

            # 割り当て後の状態を保存
            self.history.append((self.centroids.copy(), current_labels, f"Iter {i+1}: Assignment"))

            # --- Step 2: 更新 (Update) ---
            # 所属（色）は変えず、重心の位置だけ動かす
            old_centroids = self.centroids.copy()
            for k in range(self.K):
                points = self.X[current_labels == k]
                if len(points) > 0:
                    self.centroids[k] = points.mean(axis=0)

            # 更新後の状態を保存
            self.history.append((self.centroids.copy(), current_labels, f"Iter {i+1}: Update Centroids"))

            # 収束判定 (重心が動かなかったら終了)
            if np.allclose(old_centroids, self.centroids):
                print(f"収束しました (Iteration {i+1})")
                break

    def _assign_labels(self):
        # 各データと重心の距離計算
        distances = np.linalg.norm(self.X[:, np.newaxis] - self.centroids, axis=2)
        return np.argmin(distances, axis=1)

# モデルの実行
kmeans_anim = KMeansDetailedHistory(X_manual, K=2, initial_centroids=centroids_manual)
kmeans_anim.fit()

# --- アニメーションの作成 ---
fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    ax.clear()
    centroids, labels, title = kmeans_anim.history[frame]

    # データ点のプロット（所属ごとに色分け）
    # クラスタA(0): 青, クラスタB(1): オレンジ
    colors = ['blue' if l == 0 else 'orange' for l in labels]
    ax.scatter(X_manual[:, 0], X_manual[:, 1], c=colors, s=150, alpha=0.6, label='Data Points')

    # データ点に番号を振る
    for i, txt in enumerate(['x1', 'x2', 'x3', 'x4']):
        ax.annotate(txt, (X_manual[i, 0]+0.1, X_manual[i, 1]+0.1), fontsize=12)

    # 重心のプロット
    # クラスタA(0): 青, クラスタB(1): オレンジ
    # 重心には大きな「×」印をつける
    ax.scatter(centroids[0, 0], centroids[0, 1], c='blue', s=300, marker='X', edgecolors='black', linewidth=2, label='Centroid A')
    ax.scatter(centroids[1, 0], centroids[1, 1], c='orange', s=300, marker='X', edgecolors='black', linewidth=2, label='Centroid B')

    # 重心の軌跡を描画（オプション）
    if frame > 0:
        prev_centroids, _, _ = kmeans_anim.history[frame-1]
        # 前回の位置から現在の位置へ矢印
        for k in range(len(centroids)):
             ax.arrow(prev_centroids[k, 0], prev_centroids[k, 1],
                      centroids[k, 0] - prev_centroids[k, 0],
                      centroids[k, 1] - prev_centroids[k, 1],
                      head_width=0.1, head_length=0.1, fc='gray', ec='gray', alpha=0.5)

    # タイトルと軸設定
    ax.set_title(title, fontsize=14)
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 6)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend(loc='lower right')

# アニメーション生成 (各フレームを1.5秒表示)
anim = FuncAnimation(fig, update, frames=len(kmeans_anim.history), interval=1500, repeat=True)

print("\n--- アニメーション生成中... ---")
plt.close() # 静止画プロットを閉じる
HTML(anim.to_jshtml())

### **mva_08-B**
✅  **目的**: ``scikit-learn`` ライブラリを用いてK-means法を実装し、エルボー法を用いて最適なクラスタ数 $K$ を決定するプロセスを習得しましょう。

#### 💻 **【実装・応用】 ``KMeans`` クラスとエルボー法**

実務データは多次元であり、人間の目で見て「3つのグループがある」と判断できないことがほとんどです。そこで、クラスタ内誤差平方和 (SSE: ``scikit-learn``では ``inertia_`` と呼ばれます) の推移を確認して $K$ を決定します。

詳細な仕様については、以下の公式ドキュメントを参照してください。

- ``scikit-learn`` 公式ドキュメント: ``KMeans``

#### **具体例で確認してみよう！**

4つの塊を持つ少し複雑なデータを生成し、K=1 から K=10 まで試行して最適なクラスタ数を探索します。


In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. データの生成（4つの中心を持つデータ）
# ==========================================
# n_samples=300, centers=4, cluster_std=0.8, random_state=0
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=0)

# ==========================================
# 2. エルボー法によるKの探索
# ==========================================
sse = [] # Sum of Squared Errors (Inertia)
k_range = range(1, 11)

for k in k_range:
    # n_init='auto' または 10 を指定して初期値依存のリスクを減らす
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(X)

    # inertia_ 属性にSSEが格納されている
    sse.append(kmeans.inertia_)

# 結果の可視化
plt.figure(figsize=(8, 5))
plt.plot(k_range, sse, marker='o', color='b')
plt.xlabel('Number of clusters (K)')
plt.ylabel('SSE (Inertia)')
plt.title('Elbow Method for Optimal K')
plt.xticks(k_range)
plt.grid(True)

# エルボーポイントの目安（例えば K=4 付近）
plt.annotate('Elbow Point?', xy=(4, sse[3]), xytext=(5, sse[3]+500),
             arrowprops=dict(facecolor='black', shrink=0.05))

plt.show()

# ==========================================
# 3. 異なる K の結果比較 (K=3, 4, 5)
# ==========================================
# エルボー法の妥当性を確認するため、最適な K とその前後の結果を可視化して比較します。

k_candidates = [3, 4, 5]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, k in enumerate(k_candidates):
    # クラスタリング実行
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    y_pred = kmeans.fit_predict(X)
    sse_val = kmeans.inertia_

    # 可視化
    ax = axes[i]
    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y_pred, palette='viridis', s=50, alpha=0.8, legend=False, ax=ax)

    # 重心のプロット
    centers = kmeans.cluster_centers_
    ax.scatter(centers[:, 0], centers[:, 1], c='red', s=200, marker='X', edgecolors='white', linewidth=2)

    ax.set_title(f"K = {k} (SSE = {sse_val:.0f})")
    ax.grid(True)

plt.tight_layout()
plt.show()